# 03 — XGBoost Training (v2)

**Purpose:** Train h=1 model, compare to WMA on identical rows.

**Run after notebook 02.** Assumes X_train, y_train, X_test, y_test, wma_test_inputs, FEATURE_COLS in memory.

**v2 changes:**
- Rolling averages NOT in features (leakage removed)
- WMA on EXACT same row indices as model (fair comparison)
- Separate model per horizon


## Cell 1 — MLflow setup

In [ ]:
import numpy as np
import pandas as pd
import xgboost as xgb
import mlflow
import mlflow.xgboost
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pathlib import Path

ML_ROOT = Path.cwd().parent
mlflow.set_tracking_uri(f"sqlite:///{ML_ROOT}/mlflow.db")
mlflow.set_experiment("demand_forecasting")
print("MLflow ready:", mlflow.get_tracking_uri())

## Cell 2 — Hyperparameters

In [ ]:
XGB_PARAMS = {
    "n_estimators": 400,
    "max_depth": 6,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "min_child_weight": 5,
    "tree_method": "hist",
    "random_state": 42,
    "n_jobs": -1,
}
print(XGB_PARAMS)

## Cell 3 — Train h=1

Production train.py runs h=1..7 on full data sequentially.
Notebook uses 300-product sample — fast validation only.

In [ ]:
model_h1 = xgb.XGBRegressor(**XGB_PARAMS)
model_h1.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50,
)
print("Training complete.")

## Cell 4 — Fair evaluation

wma_test_inputs comes from df_sorted.loc[valid, ...] in notebook 02 — EXACT same rows as X_test.

In [ ]:
model_preds = model_h1.predict(X_test)

wma_preds = (
    0.6 * wma_test_inputs["last_7_day_avg"].fillna(0)
    + 0.3 * wma_test_inputs["last_30_day_avg"].fillna(0)
    + 0.1 * wma_test_inputs["last_60_day_avg"].fillna(0)
).values

def metrics(y_true, y_pred, label):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    print(f"{label:20s}  MAE={mae:.4f}  RMSE={rmse:.4f}")
    return mae

y_vals    = y_test.values
wma_mae   = metrics(y_vals, wma_preds,   "WMA baseline")
model_mae = metrics(y_vals, model_preds, "XGBoost h=1")
improvement = (wma_mae - model_mae) / wma_mae * 100
print(f"
Improvement: {improvement:+.1f}%")
print(f"Evaluated on {len(y_test):,} rows")

## Cell 5 — Feature importance

lag_1_qty should rank #1 or #2. New features should appear in top 5.

In [ ]:
import matplotlib.pyplot as plt

importance = model_h1.get_booster().get_fscore()
imp_df = (
    pd.DataFrame(list(importance.items()), columns=["feature", "score"])
    .sort_values("score", ascending=False)
)
print(imp_df.to_string(index=False))

imp_df.plot.barh(x="feature", y="score", figsize=(8, 5), legend=False)
plt.title("Feature importance h=1")
plt.tight_layout()
plt.show()

## Cell 6 — Log to MLflow

In [ ]:
with mlflow.start_run(run_name="notebook_v2_h1") as run:
    mlflow.log_params(XGB_PARAMS)
    mlflow.log_param("features", str(FEATURE_COLS))
    mlflow.log_metric("h1_model_mae",   model_mae)
    mlflow.log_metric("h1_wma_mae",     wma_mae)
    mlflow.log_metric("h1_improvement", improvement)
    mlflow.log_metric("train_rows",     len(X_train))
    mlflow.log_metric("test_rows",      len(X_test))
    mlflow.xgboost.log_model(model_h1, "model_h1")
    run_id = run.info.run_id
print(f"Run logged. ID: {run_id}")

## Cell 7 — h=1, h=3, h=7 comparison

Expected: h=1 MAE < h=3 MAE < h=7 MAE.

In [ ]:
# Requires df_clean, df_sorted, cutoff, FEATURE_COLS from notebook 02
for h in [1, 3, 7]:
    tgt  = df_sorted.groupby("product_id")["quantity_sold"].shift(-h)
    valid = tgt.notna()
    Xh   = df_sorted.loc[valid, FEATURE_COLS].fillna(0).astype("float32")
    yh   = tgt[valid].astype("float32")
    dh   = df_sorted.loc[valid, "date"]
    wh   = df_sorted.loc[valid, ["last_7_day_avg","last_30_day_avg","last_60_day_avg"]]
    Xtr, ytr = Xh[dh<=cutoff], yh[dh<=cutoff]
    Xte, yte = Xh[dh>cutoff],  yh[dh>cutoff]
    wte = wh[dh>cutoff]
    m = xgb.XGBRegressor(**XGB_PARAMS)
    m.fit(Xtr, ytr, eval_set=[(Xte,yte)], verbose=False)
    mp   = m.predict(Xte)
    wmap = (0.6*wte["last_7_day_avg"].fillna(0)+0.3*wte["last_30_day_avg"].fillna(0)+0.1*wte["last_60_day_avg"].fillna(0)).values
    m_mae = mean_absolute_error(yte.values, mp)
    w_mae = mean_absolute_error(yte.values, wmap)
    print(f"h={h:2d}  WMA={w_mae:.4f}  Model={m_mae:.4f}  delta={(w_mae-m_mae)/w_mae*100:+.1f}%")